# Prédiction de matchs LoL Pro — Exploration des données
**Dataset** : Oracle's Elixir 2025  
**Approche** : Prédiction à différents checkpoints temporels (10, 15, 20, 25 min)  
**Objectif** : Simuler une prédiction en temps réel

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

df_raw = pd.read_csv('../data/2025_LoL_esports_match_data_from_OraclesElixir.csv', low_memory=False)
print(f'Shape brut : {df_raw.shape}')

## 1. Filtrage — lignes équipe, matchs complets

In [ ]:
df = df_raw[
    (df_raw['participantid'].isin([100, 200])) &
    (df_raw['datacompleteness'] == 'complete')
].copy()

df['side'] = (df['side'] == 'Blue').astype(int)

print(f'Matchs complets : {df["gameid"].nunique()}')
print(f'Lignes équipe   : {len(df)}')
print(f'Taux victoire   : {df["result"].mean():.2%}')

## 2. Définition des features par checkpoint

À chaque checkpoint, on n'utilise que les informations **disponibles à cet instant**.
Les stats de fin de match sont exclues — ce serait de la triche.

In [ ]:
CONTEXT = ['side', 'firstblood']

FEATS_10 = CONTEXT + [
    'golddiffat10', 'xpdiffat10', 'csdiffat10',
    'killsat10', 'assistsat10', 'deathsat10'
]
FEATS_15 = FEATS_10 + [
    'firstdragon', 'firstherald', 'firsttower',
    'golddiffat15', 'xpdiffat15', 'csdiffat15',
    'killsat15', 'assistsat15', 'deathsat15'
]
FEATS_20 = FEATS_15 + [
    'golddiffat20', 'xpdiffat20', 'csdiffat20',
    'killsat20', 'assistsat20', 'deathsat20'
]
FEATS_25 = FEATS_20 + [
    'firstbaron',
    'golddiffat25', 'xpdiffat25', 'csdiffat25',
    'killsat25', 'assistsat25', 'deathsat25'
]

CHECKPOINTS = {'10 min': FEATS_10, '15 min': FEATS_15, '20 min': FEATS_20, '25 min': FEATS_25}
TARGET = 'result'

print('Features par checkpoint :')
for name, feats in CHECKPOINTS.items():
    n = len(df[feats + [TARGET]].dropna()) // 2
    print(f'  @{name} : {len(feats):2d} features | {n} matchs')

## 3. Distributions @10 min selon le résultat

In [ ]:
df10 = df[FEATS_10 + [TARGET]].dropna()

key_feats = ['golddiffat10', 'xpdiffat10', 'csdiffat10', 'killsat10']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, feat in zip(axes, key_feats):
    for result, color, label in [(0, '#e74c3c', 'Défaite'), (1, '#2ecc71', 'Victoire')]:
        ax.hist(df10[df10[TARGET] == result][feat], bins=30,
                alpha=0.6, color=color, label=label, density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)

plt.suptitle('Distributions @10 min (victoire vs défaite)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Corrélations avec la victoire par checkpoint

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 7))

for ax, (name, feats) in zip(axes, CHECKPOINTS.items()):
    subset = df[feats + [TARGET]].dropna()
    corr = subset.corr(numeric_only=True)[TARGET].drop(TARGET).sort_values(key=abs, ascending=True)
    colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in corr]
    corr.plot(kind='barh', ax=ax, color=colors, edgecolor='black')
    ax.set_title(f'@{name}')
    ax.set_xlabel('Corrélation')
    ax.axvline(0, color='black', linewidth=0.8)

plt.suptitle('Corrélations avec la victoire par checkpoint', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Gold diff @10 min

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for result, color, label in [(0, '#e74c3c', 'Défaite'), (1, '#2ecc71', 'Victoire')]:
    ax.hist(df10[df10[TARGET] == result]['golddiffat10'],
            bins=40, alpha=0.6, color=color, label=label, density=True)

ax.axvline(0, color='black', linestyle='--', linewidth=1.5, label='Gold diff = 0')
ax.set_title('Gold difference @10 min selon le résultat final')
ax.set_xlabel('Gold difference @10 min')
ax.legend()
plt.tight_layout()
plt.show()

win_gold = df10[df10[TARGET]==1]['golddiffat10'].mean()
loss_gold = df10[df10[TARGET]==0]['golddiffat10'].mean()
print(f'Gold diff moyen @10 min')
print(f'  Victoire : {win_gold:+.0f}')
print(f'  Défaite  : {loss_gold:+.0f}')

## 6. Sauvegarde

In [ ]:
all_feats = list(dict.fromkeys(FEATS_25))
df_save = df[all_feats + [TARGET]].copy()
df_save.to_csv('../data/data_cleaned.csv', index=False)

print(f'Sauvegardé : {df_save.shape}')
for name, feats in CHECKPOINTS.items():
    n = len(df_save[feats + [TARGET]].dropna()) // 2
    print(f'  @{name} : {n} matchs')